# Fase 08 — Feature engineering CONTEXTUAL

**Pergunta:** que mecanismo de localização/comparável/posição relativa falta?

**Nota (pós fase 12):** índice espacial refit sobre `train` sem augmentation.

**Nota (refatoração P5):** a fase chama `run_contextual_features_pipeline` (`src/scripts/generate_features.py`)
diretamente em processo. A função lê o parquet de RAW/DERIVED do disco (gerado pela fase 07) em vez de
receber o `df` em memória, para poder rodar de forma independente da fase 07 caso o parquet já exista.

In [1]:
import sys
from pathlib import Path
ROOT = Path.cwd() if (Path.cwd() / ".git").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT))
import os
os.chdir(ROOT)

In [ ]:
from src.scripts.generate_features import run_contextual_features_pipeline

report = run_contextual_features_pipeline()
report

{'n_rows': 21612,
 'n_columns': 68,
 'new_features': ['dist_to_seattle_center',
  'comps_knn_price',
  'local_grade_percentile',
  'dist_to_nearest_train_zip',
  'comps_knn_neighbor_distance',
  'local_price_dispersion'],
 'features_contextual_parquet': 'data/trusted/features_contextual.parquet',
 'spatial_index_path': 'artifacts/spatial_index.pkl'}

In [3]:
import pandas as pd

df = pd.read_parquet(report["features_contextual_parquet"])
train = df[df["split"] == "train"]
test = df[df["split"] == "test"]
print(f"train shape: {train.shape}  test shape: {test.shape}")

train shape: (15100, 68)  test shape: (2644, 68)


In [4]:
rows = []
for c in report["new_features"]:
    r_train = train[c].corr(train["price_log"])
    r_test = test[c].corr(test["price_log"])
    rows.append({"feature": c, "r_train": r_train, "r_test": r_test, "gap": r_test - r_train})
pd.DataFrame(rows)

,feature,r_train,r_test,gap
0,dist_to_seattle_center,-0.188462,-0.190347,-0.001885
1,comps_knn_price,0.833188,0.600870,-0.232318
2,local_grade_percentile,0.435466,0.504077,0.068611
3,dist_to_nearest_train_zip,-0.019534,-0.049935,-0.030401
4,comps_knn_neighbor_distance,-0.013866,-0.050886,-0.037020
5,local_price_dispersion,0.288256,0.115093,-0.173163


In [5]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

n = len(report["new_features"])
fig, axes = plt.subplots(2, (n + 1) // 2, figsize=(5 * ((n + 1) // 2), 8))
axes = axes.ravel()
palette = ["#4C72B0", "#55A868", "#DD8452", "#C44E52", "#8172B2", "#937860"]
for ax, col, color in zip(axes, report["new_features"], palette):
    ax.scatter(train[col], train["price_log"], s=3, alpha=0.3, color=color)
    ax.set_xlabel(col)
    ax.set_title(f"r={train[col].corr(train['price_log']):.3f}")
for ax in axes[n:]:
    ax.axis("off")
plt.tight_layout()
plt.savefig("reports/figures/08_contextual.png", dpi=110)
plt.close()
print("Figura salva em reports/figures/08_contextual.png")

Figura salva em reports/figures/08_contextual.png


## Candidatas novas — dirigidas pelos achados das fases 04/08

As 3 originais seguem: `comps_knn_price` fica dependente da qualidade do índice espacial e sofre com o
holdout geográfico do split (fase 03); `dist_to_seattle_center` é um ponto fixo arbitrário. 3 novas
candidatas, todas reusando os índices já fitados em `train` (sem novo `.fit`):

- **`dist_to_nearest_train_zip`** — o diagnóstico da fase 04 (`nearest_train_zip_distance`) nunca virou
  feature real do modelo; promovido aqui por imóvel (não por zipcode agregado).
- **`comps_knn_neighbor_distance`** — distância média aos k=30 vizinhos de `comps_knn_price`, hoje
  descartada internamente. Ataca direto o gap -0,232 (train→test) que a célula anterior mediu para
  `comps_knn_price`.
- **`local_price_dispersion`** — desvio-padrão do `price_log` entre os mesmos k=30 vizinhos, sinal de
  heterogeneidade/incerteza do mercado local.

Registradas em `data/processing/feature_registry.csv` como `candidate` (aguardando ablation formal da
fase 09).

## Análise

Execução em processo confirma números idênticos aos da versão pré-reestruturação para as 3 features
originais: TRAIN com 15.100 linhas e TEST com 2.644 linhas. Com as 3 candidatas novas, `n_columns` sobe
de 62 para 68, gravadas em `data/trusted/features_contextual.parquet`.

`comps_knn_price` continua a mais forte (r=0,833 → 0,601, gap -0,232) e `local_grade_percentile`
estável (0,436 → 0,504). Das 3 novas, resultado misto e **reportado sem inflar**: `local_price_dispersion`
tem sinal isolado real (r=0,289 em TRAIN, 0,115 em TEST — cai bastante mas continua positivo);
`dist_to_nearest_train_zip` (r=-0,020 → -0,050) e `comps_knn_neighbor_distance` (r=-0,014 → -0,051) têm
correlação **isolada** com `price_log` próxima de zero — esperado e não é sinal de que a ideia falhou:
essas duas não foram desenhadas para prever preço diretamente, foram desenhadas como sinal de
confiança/cobertura pra modular `comps_knn_price` (ex: descontar o comp quando o vizinho mais próximo
está longe). Um modelo baseado em árvore (XGBoost) captura esse tipo de interação não-linear que a
correlação de Pearson marginal não enxerga — a fase 09 (ablation, com o modelo real) é o teste que
decide se a ideia se sustenta, não esta correlação bruta.

![contextual](../reports/figures/08_contextual.png)

## Decisão

`data/trusted/features_contextual.parquet` e `artifacts/spatial_index.pkl` gerados (índice fit só sobre
`train`; `spatial_index.pkl` agora também carrega `train_zip_index`, reusado por
`dist_to_nearest_train_zip`). Todas as 6 features (3 originais + 3 novas) seguem `candidate` em
`data/processing/feature_registry.csv`. Decisão de promoção fica para a fase 09 (ablation) — reforço
específico pras 3 novas: `comps_knn_neighbor_distance`/`dist_to_nearest_train_zip` só fazem sentido
testadas em **interação** com `comps_knn_price`, não isoladas.

## Artefatos

`data/trusted/features_contextual.parquet`, `artifacts/spatial_index.pkl`,
`reports/figures/08_contextual.png`, `data/processing/feature_registry.csv` (3 linhas novas),
`src/features/comparable.py` (`apply_comps_knn_neighbor_distance`, `apply_local_price_dispersion`),
`src/features/neighborhood.py` (`fit_train_zip_index`, `apply_dist_to_nearest_train_zip`)